In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch, os, subprocess
os.environ['TORCH_HOME'] = '/content/torch_cache'
torch.hub.set_dir('/content/torch_cache/hub')

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO CUDA - go to Runtime > Change Runtime Type > GPU: T4')

!pip install -q fastapi 'uvicorn[standard]' websockets timm ultralytics opencv-python-headless transformers accelerate xformers

!pip install -q --use-deprecated=legacy-resolver depth-anything-3 || pip install -q --no-build-isolation depth-anything-3 || echo 'WARNING: depth-anything-3 failed to install. DA3-Large will not be available. Server falls back to DAV2.'

subprocess.run(['wget', '-q', '-O', '/tmp/cloudflared.deb',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'],
    check=True)
subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], capture_output=True, check=True)
print('cloudflared installed.')

print('Dependencies ready.')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, shutil

MODEL_RUN = 'run6_alt'
MODEL_FILENAME = f'nova_yolo_{MODEL_RUN}_best.pt'

DRIVE_FOLDER = '/content/drive/MyDrive/Server Files'
SERVER_DIR   = '/content/nova_server'

if not os.path.exists(DRIVE_FOLDER):
    raise FileNotFoundError(f'Drive folder not found: {DRIVE_FOLDER}')

os.makedirs(SERVER_DIR, exist_ok=True)

init_path = os.path.join(SERVER_DIR, '__init__.py')
if not os.path.exists(init_path):
    open(init_path, 'w').close()
    print('  created __init__.py')

for fname in ['depth_utils.py', 'alert_logic.py', 'nova_server.py']:
    src = os.path.join(DRIVE_FOLDER, fname)
    dst = os.path.join(SERVER_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  copied {fname}')
    else:
        print(f'  MISSING: {fname}')

model_src = os.path.join(DRIVE_FOLDER, MODEL_FILENAME)
model_dst = f'/content/{MODEL_FILENAME}'
if os.path.exists(model_src):
    shutil.copy(model_src, model_dst)
    print(f'  copied model ({os.path.getsize(model_dst)/1e6:.0f} MB): {MODEL_FILENAME}')
else:
    print(f"  MISSING: {MODEL_FILENAME}  (check Drive folder)")

print('Done. Files in /content/nova_server:')
for f in sorted(os.listdir(SERVER_DIR)):
    print(f'  {f}')

In [ ]:
import os

os.makedirs('/content/drive/MyDrive/Server Files/hf_cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/Server Files/hf_cache'
print('HF cache:', os.environ['HF_HOME'])

os.environ['REGISTRY_URL'] = 'https://nova-url-registry-production.up.railway.app'
os.environ['REGISTRY_TOKEN'] = ''

model_path = f'/content/{MODEL_FILENAME}'
if os.path.exists(model_path):
    os.environ['NOVA_MODEL_PATH'] = model_path
    print(f'Model: {model_path} ({os.path.getsize(model_path)/1e6:.0f} MB)')
else:
    print(f'Model not found at {model_path} - run Cell 2 first')

os.environ.pop('NGROK_TOKEN', None)

os.environ['HF_HUB_DISABLE_XET'] = '1'
print('HF_HUB_DISABLE_XET=1 (avoids Colab Xet-download stalls)')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = ''
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN set from Colab Secrets - faster/authenticated downloads.')
else:
    print('No HF_TOKEN secret found - DA3-Large download may be slow/throttled on first run.')
    print('  Add one: Colab sidebar -> key icon -> Add secret -> name HF_TOKEN -> enable notebook access.')

print('Config ready.')


In [ ]:
import subprocess, threading, time, sys, os, re, requests

ok = True
for path, label in [
    ('/content/nova_server/__init__.py',    '__init__.py'),
    ('/content/nova_server/depth_utils.py', 'depth_utils.py'),
    ('/content/nova_server/alert_logic.py', 'alert_logic.py'),
    ('/content/nova_server/nova_server.py', 'nova_server.py'),
    (f'/content/{MODEL_FILENAME}',          MODEL_FILENAME),
]:
    if os.path.exists(path):
        print(f'  ok  {label}')
    else:
        print(f'  MISSING: {label}')
        ok = False

if not ok:
    raise RuntimeError('Fix missing items above before starting server.')

def run_server():
    p = subprocess.Popen(
        ['python', '-u', '-m', 'uvicorn', 'nova_server.nova_server:app',
         '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
        cwd='/content',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )
    for line in p.stdout:
        print(line.rstrip())

threading.Thread(target=run_server, daemon=True).start()
print('Server starting...')
print(f'  Model: {MODEL_RUN}. YOLO-640 (fp16) + DA3-Large Metric depth (autocast fp16, primary).')
print('  Normal startup: ~60s. First run: DA3-Large downloads to Drive hf_cache/ (~2-5 min extra).')

def start_cloudflare_tunnel():
    for attempt in range(240):
        time.sleep(3)
        try:
            if requests.get('http://localhost:8000/health', timeout=2).ok:
                break
        except Exception:
            pass
    else:
        print('[tunnel] Server did not become ready in 12 min - cloudflare tunnel aborted.')
        return

    print('\n[tunnel] Server ready - opening Cloudflare tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        line = line.rstrip()
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            https_url = m.group(0)
            wss_url = https_url.replace('https://', 'wss://') + '/ws'
            print(f'\n[NOVA] PUBLIC WSS URL: {wss_url}')
            try:
                reg = os.environ.get('REGISTRY_URL', '')
                if reg:
                    r = requests.post(f'{reg}/server-url', params={'url': wss_url, 'token': os.environ.get('REGISTRY_TOKEN', '')}, timeout=10)
                    if r.ok:
                        print('[NOVA] Registered with Railway registry - Android will auto-connect.')
                    else:
                        print(f'[NOVA] Registry error {r.status_code}')
            except Exception as e:
                print(f'[NOVA] Registry unreachable: {e}')
        elif 'ERR' in line or 'error' in line.lower():
            print(f'[tunnel] {line}')

threading.Thread(target=start_cloudflare_tunnel, daemon=True).start()

print('Polling for server readiness (up to 12.5 min - DA3-Large may still be downloading)...')
for i in range(150):
    time.sleep(5)
    try:
        h = requests.get('http://localhost:8000/health', timeout=3).json()
        print(f'\nServer UP after {(i + 1) * 5}s: {h}')
        print('Android auto-connects via Railway registry.')
        break
    except Exception:
        if (i + 1) % 6 == 0:
            print(f'  still starting... ({(i + 1) * 5}s elapsed)')
else:
    print('\nNot ready after 5 min. Check server logs above for errors.')
    print('Manual check once ready:')
    print('  import requests; print(requests.get("http://localhost:8000/health").json())')

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopped.')